# Fine-Tuning Qwen3.5-0.8B (QLoRA) — Chatbot Medis ID (Pivot 4)

Dataset **ID native-only** v2.1-remediated (`Data/processed_id/`, 30003/2997/2998). Toolchain **Unsloth + TRL + PEFT**, QLoRA 4-bit, dijalankan di **Kaggle/Colab T4 (single GPU)**.

**Cara pakai:** jalankan sel berurutan. Mulai `RUN_MODE="pilot"` (sel Konfigurasi) → baca **GATE** → kalau PASS, set `RUN_MODE="full"`, Restart & run. Detail + troubleshooting di `README_TRAIN.md`.

> Validasi dataset terpisah: `preprocessing/validate_dataset_for_training.py` (harus VERDICT=PASS sebelum notebook ini).

## 1. Install (Unsloth/TRL/PEFT/bnb) — lalu **Restart jika diminta**

In [ ]:
%%capture
# Cell install RESMI Unsloth dengan versi PINNED (menyelesaikan konflik
# torchaudio/_LazyModule/no-quant_state pada model VLM Qwen3.5 & Gemma 4).
# JANGAN ubah versi. Sumber: notebook resmi Unsloth Qwen3.5/Gemma 4 (Vision).
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --upgrade transformers
!pip install --no-deps trl==0.22.2
!pip install torchcodec
import torch; torch._dynamo.config.recompile_limit = 64

## 2. Environment (single GPU, seed)

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")
import torch, random, numpy as np, json, glob, math
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# --- A100 freebie: percepat matmul (aman, tidak mempengaruhi resume) ---
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 3. Konfigurasi — set `RUN_MODE` di sini

`pilot` = subset 1500/200, ~250 step (cek sinyal). `full` = data penuh + EarlyStopping. Hyperparam T4-safe; `RESPONSE_PART` menyertakan scaffold `<think></think>` (lihat sel SELF-CHECK).

In [ ]:
# ====================== IDENTITAS MODEL ======================
MODEL_ID          = "unsloth/Qwen3.5-0.8B"            # model TEKS ~0.8B (priority)
MODEL_ID_FALLBACK = "unsloth/Qwen3.5-0.8B"   # cadangan bila id utama gagal
# (Qwen3.5-0.8B = LLM teks -> FastLanguageModel. Varian 2B di notebook lain = VLM.)

# ====================== RUN MODE ======================
# "pilot" = cek sinyal cepat (subset, ~250 step). "full" = training penuh.
# MULAI dari "pilot", baca GATE, baru ganti ke "full".
RUN_MODE = "full"      # "pilot" | "full"

# ====================== HYPERPARAMETER (T4-safe) ======================
MAX_SEQ_LENGTH = 1024            # p99 dataset ~634, max 1344 (hanya 7 train >1024 -> aman)
LOAD_IN_4BIT   = True            # QLoRA
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LORA_TARGET = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

LEARNING_RATE = 1e-4             # ikut notebook lama (2e-4 terbukti terlalu agresif)
EPOCHS        = 3                # full-mode; + EarlyStopping(patience=2)
WARMUP_RATIO  = 0.10
WEIGHT_DECAY  = 0.01
PER_DEVICE_BATCH = 2             # T4 6-16GB-safe utk 0.8B 4-bit, seq 1024
GRAD_ACCUM       = 8             # effective batch = 16  (TUNABLE; lihat TROUBLESHOOTING)
MAX_GRAD_NORM    = 1.0           # grad clipping (anti loss meledak)

# ====================== FORMAT TURN (ChatML Qwen) ======================
# Template Qwen3.5 menyisipkan <think>\n\n</think> kosong setelah 'assistant\n'
# walau enable_thinking=False. RESPONSE_PART menyertakannya supaya
# train_on_responses_only HANYA melatih jawaban medis (train==inferensi).
INSTRUCTION_PART = "<|im_start|>user\n"
RESPONSE_PART    = "<|im_start|>assistant\n<think>\n\n</think>\n\n"

# ====================== PILOT SUBSET ======================
PILOT_TRAIN_N, PILOT_VAL_N, PILOT_MAX_STEPS = 1500, 200, 250

# ====================== OUTPUT ======================
ADAPTER_DIR = "/content/drive/MyDrive/Aries/Fine-Tune SLM for Medical Chatbot/outputs/checkpoints/qwen35-0.8b-train"
print(f"RUN_MODE={RUN_MODE} | eff_batch={PER_DEVICE_BATCH*GRAD_ACCUM} | seq={MAX_SEQ_LENGTH}")

## 4. Lokasi dataset beku (Kaggle/Colab/lokal)

Upload `train/val/test.jsonl` sbg Kaggle dataset atau ke Drive — lihat `README_TRAIN.md`. Sel ini mencari otomatis (atau set env `DATA_DIR`).

In [ ]:
# Cari dataset beku (Kaggle input / Colab Drive / lokal). Lihat README_TRAIN.md utk upload.
def _first_existing(paths):
    for p in paths:
        if p and os.path.exists(os.path.join(p, "train.jsonl")):
            return p
    return None

CANDIDATES = [
    os.environ.get("DATA_DIR", ""),
    "/content/drive/MyDrive/Aries/Fine-Tune SLM for Medical Chatbot/Data/processed_id",
    "../Data/processed_id", "Data/processed_id",
]
# Colab: mount Drive bila perlu
if any("drive/MyDrive" in c for c in CANDIDATES) and not os.path.exists("/content/drive"):
    try:
        from google.colab import drive; drive.mount("/content/drive")
    except Exception:
        pass
DATA_DIR = _first_existing(CANDIDATES)
assert DATA_DIR, ("Dataset tak ketemu. Set env DATA_DIR atau upload "
                  "train/val/test.jsonl. Lihat README_TRAIN.md.")
os.makedirs(ADAPTER_DIR, exist_ok=True)
print("DATA_DIR =", DATA_DIR)

## 5. Load model 4-bit (FastLanguageModel)

Qwen3.5-0.8B = model **teks** → `FastLanguageModel` (bukan VLM). `dtype=None` → T4 auto fp16.

In [ ]:
from unsloth import FastVisionModel

def _load(model_name):
    return FastVisionModel.from_pretrained(
        model_name                 = model_name,
        load_in_4bit               = LOAD_IN_4BIT,
        use_gradient_checkpointing = "unsloth",
        dtype                      = None,
        text_only=True,
    )

try:
    model, tokenizer = _load(MODEL_ID)
    print("Loaded:", MODEL_ID)
except Exception as e:
    print("Gagal load (", repr(e)[:140], ") -> fallback:", MODEL_ID_FALLBACK)
    model, tokenizer = _load(MODEL_ID_FALLBACK)

## 6. LoRA adapter (QLoRA)

**Catat `print_trainable_parameters()`** (BAB III).

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = False,   # TEXT-ONLY (matikan menara vision)
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    bias           = "none",
    target_modules = "all-linear",
    random_state   = SEED,
    use_rslora     = False,
    loftq_config   = None,
)
model.print_trainable_parameters()   # CATAT angka ini utk BAB III

## 7. Dataset → teks (chat template Qwen, thinking OFF)

In [ ]:
from datasets import load_dataset

def render_chat(messages, add_generation_prompt=False):
    # enable_thinking=False -> Qwen tetap sisipkan <think></think> kosong (di-mask via RESPONSE_PART)
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False,
                   add_generation_prompt=add_generation_prompt, enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False,
                   add_generation_prompt=add_generation_prompt)

def formatting_func(ex):
    return {"text": render_chat(ex["messages"], add_generation_prompt=False)}

train_ds = load_dataset("json", data_files=os.path.join(DATA_DIR, "train.jsonl"), split="train")
val_ds   = load_dataset("json", data_files=os.path.join(DATA_DIR, "val.jsonl"),   split="train")

if RUN_MODE == "pilot":          # ← ganti dari "full" ke "pilot"
    train_ds = train_ds.shuffle(seed=SEED).select(range(min(PILOT_TRAIN_N, len(train_ds))))
    val_ds   = val_ds.shuffle(seed=SEED).select(range(min(PILOT_VAL_N, len(val_ds))))

train_ds = train_ds.map(formatting_func, remove_columns=train_ds.column_names)
val_ds   = val_ds.map(formatting_func,   remove_columns=val_ds.column_names)
print(f"RUN_MODE={RUN_MODE} | train={len(train_ds)} | val={len(val_ds)}")

## 8. SELF-CHECK decode (WAJIB dilihat sebelum train)

Membuktikan token **konten pertama jawaban** ikut dipelajari (bukan ke-mask / off-by-one) dan scaffold `<think></think>` membuat **train == inferensi**. Identik dgn Deliverable 1 (`validate_dataset_for_training.py`).

In [ ]:
# ===== SELF-CHECK decode (sama logika dgn validate_dataset_for_training.py) =====
# Pastikan: token KONTEN pertama jawaban IKUT dipelajari (tak ke-mask / off-by-one),
# scaffold <think></think> konsisten train==inferensi.
def _find_sub(seq, sub):
    for i in range(len(seq)-len(sub)+1):
        if seq[i:i+len(sub)] == sub: return i
    return -1

_rec   = json.loads(open(os.path.join(DATA_DIR, "train.jsonl"), encoding="utf-8").readline())
_msgs  = _rec["messages"]
_full  = render_chat(_msgs, add_generation_prompt=False)
_prompt= render_chat([m for m in _msgs if m["role"]!="assistant"], add_generation_prompt=True)
_ids   = tokenizer(_full, add_special_tokens=False)["input_ids"]
_marker= tokenizer(RESPONSE_PART, add_special_tokens=False)["input_ids"]
_pos   = _find_sub(_ids, _marker)
assert _pos >= 0, "RESPONSE_PART tak ditemukan di teks ter-render! Cek template/scaffold think."
_start = _pos + len(_marker)
_ans   = next(m["content"] for m in _msgs if m["role"]=="assistant")
_w1    = _ans.split()[0] if _ans.split() else ""
_reco  = tokenizer.decode(_ids[_start:_start+12]).lstrip()
print("first ACTIVE tokens :", [tokenizer.decode([t]) for t in _ids[_start:_start+8]])
print("jawaban kata-1 asli :", repr(_w1), "| rekonstruksi:", repr(_reco[:40]))
print("infer prompt ends   :", repr(_prompt.rstrip()[-30:]))
assert _reco.lower().startswith(_w1[:6].lower()), "OFF-BY-ONE: token jawaban pertama ke-mask/terpotong!"
assert _prompt.rstrip().endswith("</think>"), "Inferensi TIDAK berakhir </think> -> train != infer!"
print("OK decode-check: jawaban medis pertama dipelajari, train==inferensi konsisten.")

## 9. Konfigurasi training (SFT + train_on_responses_only)

`train_on_responses_only` → loss **hanya pada jawaban medis** (scaffold think masuk `RESPONSE_PART` → ikut ter-mask). `packing=False` (wajib). T4 → `fp16=True`, `max_grad_norm=1.0`. Vocab Qwen ~248k besar → Unsloth otomatis patch cross-entropy (tanpa `compute_loss_func` manual).

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback
from unsloth.chat_templates import train_on_responses_only
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
USE_BF16 = torch.cuda.is_bf16_supported()   # T4 -> False -> fp16

# cadence & durasi tergantung RUN_MODE
if RUN_MODE == "full":
    steps_kw = dict(num_train_epochs=EPOCHS, eval_steps=500, save_steps=500)
else:  # pilot
    steps_kw = dict(max_steps=PILOT_MAX_STEPS, num_train_epochs=1,
                    eval_steps=50, save_steps=50)

cfg = SFTConfig(
    output_dir                  = ADAPTER_DIR,
    per_device_train_batch_size = PER_DEVICE_BATCH,
    per_device_eval_batch_size  = 4,
    gradient_accumulation_steps = GRAD_ACCUM,
    warmup_ratio                = WARMUP_RATIO,
    learning_rate               = LEARNING_RATE,
    lr_scheduler_type           = "cosine",
    weight_decay                = WEIGHT_DECAY,
    optim                       = "adamw_8bit",
    fp16                        = not USE_BF16,    # T4 -> fp16
    bf16                        = USE_BF16,
    max_grad_norm               = MAX_GRAD_NORM,   # grad clipping
    logging_steps               = 10,
    eval_strategy               = "steps",
    save_strategy               = "steps",
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    seed                        = SEED, data_seed = SEED,
    dataset_text_field          = "text",
    max_length                  = MAX_SEQ_LENGTH,
    packing                     = False,           # WAJIB utk train_on_responses_only
    report_to                   = "none",
    **steps_kw,
)

_cb = [] if RUN_MODE == "full" else [EarlyStoppingCallback(2, 0.001)]
_kw = dict(model=model, train_dataset=train_ds, eval_dataset=val_ds, args=cfg, callbacks=_cb)
try:
    trainer = SFTTrainer(processing_class=tokenizer, **_kw)
except TypeError:
    trainer = SFTTrainer(tokenizer=tokenizer, **_kw)

# loss HANYA pada jawaban medis (scaffold <think> ikut RESPONSE_PART -> ter-mask)
trainer = train_on_responses_only(trainer,
    instruction_part=INSTRUCTION_PART, response_part=RESPONSE_PART)

# Vocab Qwen ~248k (besar): Unsloth otomatis mem-patch cross-entropy (fused) -> hemat VRAM,
# TIDAK perlu compute_loss_func manual. Jika muncul OOM di logits, kurangi seq/batch.
print("trainer siap. RUN_MODE =", RUN_MODE, "| steps_kw =", steps_kw)

In [ ]:
import os, glob, shutil
for ck in glob.glob(os.path.join(ADAPTER_DIR, "checkpoint-*")):
    if not os.path.exists(os.path.join(ck, "trainer_state.json")):
        print("hapus (tidak lengkap):", ck)
        shutil.rmtree(ck)
print("sisa:", sorted(glob.glob(os.path.join(ADAPTER_DIR, "checkpoint-*"))))

## 10. Train (resume-aware)

In [ ]:
# Resume bila checkpoint ada (tahan disconnect Colab/Kaggle).
_ckpts = glob.glob(os.path.join(ADAPTER_DIR, "checkpoint-*"))
_resume = len(_ckpts) > 0
print("resume:", _resume, f"({len(_ckpts)} checkpoint)")
trainer_stats = trainer.train(resume_from_checkpoint=_resume or None)
print("training selesai.")

In [ ]:
!pip install -q langdetect
import numpy as np   # jaga-jaga, GATE pakai np.isfinite

## 11. GATE (setelah PILOT) — PASS/STOP

Cek: loss turun (mean 20% akhir ≤ 20% awal − 0.15) & tak meledak/NaN; 10 generasi: ≥9/10 Indonesia (**anti language-mixing** — cacat utama Gemma3-1B dulu), tidak degeneratif. Kalau **STOP** → `README_TRAIN.md` › TROUBLESHOOTING.

In [ ]:
# ===== GATE (jalankan setelah PILOT). Cetak PASS / STOP. =====
import re as _re
from langdetect import detect, DetectorFactory; DetectorFactory.seed = 42

logs = trainer.state.log_history
losses = [l["loss"] for l in logs if "loss" in l]
def _mean(x): return sum(x)/len(x) if x else float("nan")
k = max(1, len(losses)//5)
loss_start, loss_end = _mean(losses[:k]), _mean(losses[-k:])
loss_max = max(losses) if losses else float("nan")
B2_trend = (loss_end <= loss_start - 0.15)
B2_noexplode = (loss_max <= 2*losses[0]) if losses else False
B1_stable = all(np.isfinite(l) for l in losses) and len(losses) > 0
print(f"[B1] stabil(no NaN/Inf): {B1_stable}")
print(f"[B2] loss {loss_start:.3f} -> {loss_end:.3f} (turun>=0.15: {B2_trend}; tak meledak: {B2_noexplode})")

# B3 generasi: 10 prompt tetap dari test
FastVisionModel.for_inference(model)

# Patch: Qwen3.5 text_only -> config.architectures None bikin deteksi VLM crash. Set ke CausalLM.
if not getattr(model.config, "architectures", None):
    model.config.architectures = ["Qwen3ForCausalLM"]

test_ds = load_dataset("json", data_files=os.path.join(DATA_DIR, "test.jsonl"), split="train")
test_ds = load_dataset("json", data_files=os.path.join(DATA_DIR, "test.jsonl"), split="train")
gens = []
id_ok = degen = 0
for i in range(10):
    msgs = [m for m in test_ds[i]["messages"] if m["role"] != "assistant"]
    p = render_chat(msgs, add_generation_prompt=True)
    inp = tokenizer(p, return_tensors="pt").to(model.device)
    out = model.generate(**inp, max_new_tokens=200, do_sample=False, no_repeat_ngram_size=3,
                         repetition_penalty=1.1, pad_token_id=tokenizer.pad_token_id)
    txt = tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    gens.append((msgs[-1]["content"], txt))
    # bahasa
    try:
        if detect(txt[:500]) == "id": id_ok += 1
    except Exception:
        pass
    # degeneratif: >30% 4-gram berulang
    toks = txt.split()
    grams = [tuple(toks[j:j+4]) for j in range(len(toks)-3)]
    rep = (1 - len(set(grams))/len(grams)) if grams else 0
    if rep > 0.30: degen += 1
B3a_lang = id_ok >= 9
B3b_degen = degen == 0
print(f"[B3a] Indonesia {id_ok}/10 (anti language-mixing: {B3a_lang})")
print(f"[B3b] non-degeneratif {10-degen}/10 (no >30% 4-gram berulang: {B3b_degen})")

# simpan generasi
with open("pilot_generations.txt", "w", encoding="utf-8") as f:
    for q, a in gens:
        f.write(f"Q: {q}\nA: {a}\n{'-'*60}\n")
print("contoh 3 generasi:")
for q, a in gens[:3]:
    print("Q:", q[:80]); print("A:", a[:160]); print("-"*40)

PASS = B1_stable and B2_trend and B2_noexplode and B3a_lang and B3b_degen
print("\n" + "="*60)
print("GATE:", "PASS_GREEN -> boleh set RUN_MODE='full'" if PASS
      else "STOP / NEEDS_HUMAN -> lihat TROUBLESHOOTING di README_TRAIN.md")
print("="*60)
FastVisionModel.for_training(model)

## 12. Simpan adapter + generasi

In [ ]:
# Simpan adapter LoRA (kecil) utk inspeksi/arsip. (Merge 16-bit dilakukan saat eval.)
print("best checkpoint:", trainer.state.best_model_checkpoint)
print("best eval_loss :", trainer.state.best_metric)
model.save_pretrained(ADAPTER_DIR); tokenizer.save_pretrained(ADAPTER_DIR)
print("adapter ->", ADAPTER_DIR, "| 10 generasi -> pilot_generations.txt")

## 13. (Placeholder) Eval token-F1 + ROUGE-L (val)

EM/MCQA di-DROP (Pivot-4 native-only) → metrik open-ended **token-F1 + ROUGE-L**. Sel siap pakai setelah training penuh.

In [ ]:
# ===== (PLACEHOLDER) eval token-F1 + ROUGE-L pada val — siap pakai setelah training =====
# EM (MCQA) di-DROP utk Pivot-4 native-only; metrik = token-F1 + ROUGE-L (lihat memory).
!pip install -q rouge-score
from rouge_score import rouge_scorer
import collections

def _f1(pred, ref):
    p, r = pred.split(), ref.split()
    common = collections.Counter(p) & collections.Counter(r)
    ns = sum(common.values())
    if ns == 0 or not p or not r: return 0.0
    prec, rec = ns/len(p), ns/len(r)
    return 2*prec*rec/(prec+rec)

FastVisionModel.for_inference(model)
val_eval = load_dataset("json", data_files=os.path.join(DATA_DIR, "val.jsonl"), split="train")
N = min(100, len(val_eval))
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
f1s, rls = [], []
for i in range(N):
    msgs = val_eval[i]["messages"]
    ref = next(m["content"] for m in msgs if m["role"]=="assistant")
    p = render_chat([m for m in msgs if m["role"]!="assistant"], add_generation_prompt=True)
    inp = tokenizer(p, return_tensors="pt").to(model.device)
    out = model.generate(**inp, max_new_tokens=256, do_sample=False, no_repeat_ngram_size=3,
                         repetition_penalty=1.1, pad_token_id=tokenizer.pad_token_id)
    pred = tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    f1s.append(_f1(pred, ref)); rls.append(scorer.score(ref, pred)["rougeL"].fmeasure)
print(f"[EVAL n={N}] token-F1={sum(f1s)/N:.4f} | ROUGE-L={sum(rls)/N:.4f}")
FastVisionModel.for_training(model)

---
**Selesai.** Setelah Qwen jadi baseline sehat, model ≤1B berikutnya (Gemma 3 1B IT, Llama 3.2 1B) pakai harness/notebook serupa.

In [ ]:
FastVisionModel.for_inference(model)
q = "Apa penyebab umum demam pada anak dan kapan sebaiknya dibawa ke dokter?"

# kasih text= eksplisit biar processor nggak salah baca prompt sebagai image
inputs = tokenizer(text=render_chat([{"role":"user","content":q}], add_generation_prompt=True),
                   return_tensors="pt").to(model.device)

out = model.generate(**inputs, max_new_tokens=256, do_sample=False, no_repeat_ngram_size=3)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

In [ ]:
OUT_MERGED = "qwen35-0.8b-medical-id"

In [ ]:
# (Opsional) push hasil merge ke HuggingFace Hub. Set HF_TOKEN di Colab Secrets.
from google.colab import userdata
HF_TOKEN = None
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass
if HF_TOKEN:
    repo = "AriesDjaenuri/" + OUT_MERGED          # ganti USERNAME
    model.push_to_hub_merged(repo, tokenizer, save_method="merged_16bit", token=HF_TOKEN)
    print("Pushed ->", repo)
else:
    print("HF_TOKEN tak diset -> lewati push (opsional).")